In [ ]:
import os
import glob
import zipfile
from google.colab import drive

# ── MOUNT GOOGLE DRIVE ─────────────────────────────────────────────
drive.mount('/content/drive/')

NUM_CLASS = 2  # change this (1, 2, 3...)
CLASS_NAME = f"class_{NUM_CLASS:02d}"


# ── BASE_DIR ─────────────────────────────────────────────────────

BASE_DIR = f'/content/drive/MyDrive/dataset_ad/{CLASS_NAME}/{CLASS_NAME}'
print("BASE_DIR =", BASE_DIR)

# ── PATHS ───────────────────────────────────────────────────────
TRAIN_GOOD_DIR = os.path.join(BASE_DIR, 'train', 'good')
TEST_ANOM_DIRS = glob.glob(os.path.join(BASE_DIR, 'train', 'anomaly_*'))
GT_ANOM_DIRS   = glob.glob(os.path.join(BASE_DIR, 'ground_truth_train', 'anomaly_*'))

print("TRAIN_GOOD_DIR =", TRAIN_GOOD_DIR)
print("TEST_ANOM_DIRS =", TEST_ANOM_DIRS)
print("GT_ANOM_DIRS =", GT_ANOM_DIRS)

# ── MODEL / DINOv2 ──────────────────────────────────────────────
DINO_MODEL = 'vit_base_patch14_dinov2.lvd142m'  # or vit_small / vit_large
FEATURE_LAYERS_HL = [6, 9, 12]   # alto livello
FEATURE_LAYERS_LL = [2, 4]       # basso livello

# ── IMAGE ───────────────────────────────────────────────────────
IMG_SIZE = 224
BATCH_SIZE = 16

# ── PATCHCORE ───────────────
CORESET_RATIO = 0.05
K_NEIGHBOURS = 1


# ── OUTPUT ──────────────────────────────────────────────────────
OUTPUT_DIR = f'/content/patchcore_output/{CLASS_NAME}'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("OUTPUT_DIR =", OUTPUT_DIR)

NUM_VIEWS = 5

VIEW_NAMES = [
    'front',
    'back-down',
    'back-left',
    'back-up',
    'back-right'
]

USE_SEMANTIC_SINGLE = True
USE_SEMANTIC_MULTIVIEW = True

SEMANTIC_KNN = 1
SEMANTIC_PCA_DIM = 256

In [ ]:
import os
import glob

files = sorted(glob.glob(os.path.join(TRAIN_GOOD_DIR, '*')))

print(f"Total files: {len(files)}\n")
print("First 20 files:\n")

for i, f in enumerate(files[:20]):
    print(f"{i+1:02d} - {os.path.basename(f)}")

In [ ]:
# ── Install / upgrade dependencies ──────────────────────────────────────────
!pip install -q timm einops scikit-learn tqdm matplotlib opencv-python-headless

import os, glob, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.metrics import roc_auc_score, roc_curve
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
from sklearn.model_selection import train_test_split
from collections import defaultdict

# ── PREPROCESSING ────────────────────────────────────────────────────────────
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE),
                      interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

denorm = transforms.Compose([
    transforms.Normalize(mean=[-m/s for m, s in zip(mean, std)],
                         std=[1/s for s in std])
])


# ── MULTIVIEW GROUPS (ORDERED view1 → view5) ─────────────────────────────────
def get_obj_id(p):
    return os.path.basename(p).split('_view')[0]


all_good = glob.glob(os.path.join(TRAIN_GOOD_DIR, '*.png'))

groups = defaultdict(dict)

for p in all_good:
    oid = get_obj_id(p)
    fname = os.path.basename(p)

    # view id = view1, view2, ...
    vid = f"view{fname.split('_view')[-1].replace('.png', '')}"

    groups[oid][vid] = p


object_ids = list(groups.keys())

train_ids, val_ids = train_test_split(
    object_ids,
    test_size=0.2,
    random_state=42
)

train_groups = {oid: groups[oid] for oid in train_ids}
val_groups   = {oid: groups[oid] for oid in val_ids}


# ── MULTIVIEW DATASET ────────────────────────────────────────────────────────
class MultiViewDataset(Dataset):
    def __init__(self, groups, label, transform=None):
        self.groups = groups
        self.object_ids = list(groups.keys())
        self.label = label
        self.transform = transform
        print(f'Found {len(self.object_ids)} objects | label={label}')

    def __len__(self):
        return len(self.object_ids)

    def __getitem__(self, idx):
        oid = self.object_ids[idx]
        views = self.groups[oid]

        imgs = []

        # 🔥 FIX: strict order view1 → view5
        for i in range(1, 6):
            img = Image.open(views[f'view{i}']).convert('RGB')

            if self.transform:
                img = self.transform(img)

            imgs.append(img)

        return torch.stack(imgs, dim=0), self.label, oid


# ── FLAT DATASET (PATCHCORE SINGLE VIEW) ─────────────────────────────────────
all_good_flat = glob.glob(os.path.join(TRAIN_GOOD_DIR, '*.png'))

def get_obj_id_flat(p):
    return os.path.basename(p).split('_view')[0]

groups_flat = defaultdict(list)

for p in all_good_flat:
    groups_flat[get_obj_id_flat(p)].append(p)

object_ids_flat = list(groups_flat.keys())

train_ids_flat, val_ids_flat = train_test_split(
    object_ids_flat,
    test_size=0.2,
    random_state=42
)

train_paths = [p for oid in train_ids_flat for p in groups_flat[oid]]
val_good_paths = [p for oid in val_ids_flat for p in groups_flat[oid]]


# ── DATASETS ─────────────────────────────────────────────────────────────────
print('Loading datasets...')

train_ds = ImageFolderFlat(train_paths, label=0, transform=transform)
test_good = ImageFolderFlat(val_good_paths, label=0, transform=transform)
test_anom = ImageFolderFlat(TEST_ANOM_DIRS, label=1, transform=transform)

semantic_train_ds = MultiViewDataset(train_groups, label=0, transform=transform)
semantic_test_good = MultiViewDataset(val_groups, label=0, transform=transform)


# ── DATALOADERS ─────────────────────────────────────────────────────────────
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE,
                      shuffle=False, num_workers=2, pin_memory=True)

test_good_dl = DataLoader(test_good, batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=2, pin_memory=True)

test_anom_dl = DataLoader(test_anom, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

semantic_train_dl = DataLoader(semantic_train_ds, batch_size=BATCH_SIZE,
                               shuffle=False, num_workers=2, pin_memory=True)

semantic_test_good_dl = DataLoader(semantic_test_good, batch_size=BATCH_SIZE,
                                   shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
import timm

# ─────────────────────────────────────────────────────────────────────────────
# PATCHCORE EXTRACTOR (INTRA-IMAGE: LL + HL PATCH FEATURES)
# ─────────────────────────────────────────────────────────────────────────────
class DINOv2PatchExtractor(nn.Module):
    """
    Extracts multi-layer PATCH tokens from DINOv2.
    Used for PatchCore (local anomaly detection).

    Output per image:
        [B, N_patches, C_total]
    """

    def __init__(self, model_name=DINO_MODEL, layers=FEATURE_LAYERS_HL):
        super().__init__()

        self.model = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,
            dynamic_img_size=True
        )

        self.model.eval()

        self.layers = layers
        self._hooks = []
        self._feats = {}

        self._register_hooks()

    def _register_hooks(self):
        for layer_idx in self.layers:
            block = self.model.blocks[layer_idx - 1]

            def hook(module, inp, out, idx=layer_idx):
                # [B, N_tokens, D] → remove CLS
                self._feats[idx] = out[:, 1:, :]

            self._hooks.append(
                block.register_forward_hook(hook)
            )

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()

    @torch.no_grad()
    def forward(self, x):
        """
        x: [B, 3, H, W]
        return: [B, N_patches, D_total]
        """
        self._feats = {}

        _ = self.model(x)

        feats = torch.cat(
            [self._feats[l] for l in self.layers],
            dim=-1
        )

        feats = nn.functional.normalize(feats, dim=-1)

        return feats


# ─────────────────────────────────────────────────────────────────────────────
# SEMANTIC EXTRACTOR (NOT PATCHES → CLS TOKENS ONLY)
# ─────────────────────────────────────────────────────────────────────────────
class DINOv2SemanticExtractor(nn.Module):
    """
    Extracts CLS token per image.
    Used for:
        - multiview semantic consistency
        - object-level anomaly detection
    """

    def __init__(self, model_name=DINO_MODEL):
        super().__init__()

        self.model = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,
            dynamic_img_size=True
        )

        self.model.eval()

    @torch.no_grad()
    def forward(self, x):
        """
        x: [B, 3, H, W]
        return: [B, D]
        """
        out = self.model.forward_features(x)

        # CLS token (first token)
        cls = out[:, 0, :]

        cls = nn.functional.normalize(cls, dim=-1)

        return cls


# ─────────────────────────────────────────────────────────────────────────────
# INIT MODELS
# ─────────────────────────────────────────────────────────────────────────────
print(f'Loading {DINO_MODEL}...')

extractor_hl = DINOv2PatchExtractor(layers=FEATURE_LAYERS_HL).to(DEVICE)
extractor_ll = DINOv2PatchExtractor(layers=FEATURE_LAYERS_LL).to(DEVICE)

semantic_extractor = DINOv2SemanticExtractor().to(DEVICE)


# ─────────────────────────────────────────────────────────────────────────────
# SANITY CHECK
# ─────────────────────────────────────────────────────────────────────────────
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)

out_hl = extractor_hl(dummy)
out_ll = extractor_ll(dummy)

out_sem = semantic_extractor(dummy)

print(f'Patch feature HL shape: {out_hl.shape}')  # [B, N, D_HL]
print(f'Patch feature LL shape: {out_ll.shape}')  # [B, N, D_LL]
print(f'Semantic CLS shape: {out_sem.shape}')     # [B, D]

N_PATCHES = out_hl.shape[1]
FEAT_DIM_HL = out_hl.shape[2]
FEAT_DIM_LL = out_ll.shape[2]
SEM_DIM = out_sem.shape[1]

In [ ]:
def build_memory_bank(dataloader, extractor, device, coreset_ratio):
    memory_bank_list = []
    for imgs, labels, paths in tqdm(dataloader, desc='Extracting & coreset'):
        imgs = imgs.to(device)
        feats = extractor(imgs)  # [B, N_patches, D]
        B, N, D = feats.shape
        feats = feats.reshape(-1, D).cpu()

        # Random coreset sul batch
        K = max(1, int(feats.shape[0] * coreset_ratio))
        idx = torch.randperm(feats.shape[0])[:K]
        memory_bank_list.append(feats[idx])

    memory_bank = torch.cat(memory_bank_list, dim=0)
    print(f'Memory bank final shape: {memory_bank.shape}')
    return memory_bank

In [ ]:
def build_memory_bank_cls(dataloader, extractor, device, coreset_ratio):
    memory_bank_list = []

    for imgs, labels, paths in tqdm(dataloader, desc='Extracting CLS & coreset'):

        # imgs: [B, 5, 3, H, W] → prendiamo solo view1
        if imgs.dim() == 5:
            imgs = imgs[:, 0]  # 🔥 single-view fix

        imgs = imgs.to(device)

        with torch.no_grad():
            feats = extractor(imgs)  # [B, D]

        feats = feats.cpu()

        # coreset sul batch
        K = max(1, int(feats.shape[0] * coreset_ratio))
        idx = torch.randperm(feats.shape[0])[:K]

        memory_bank_list.append(feats[idx])

    memory_bank = torch.cat(memory_bank_list, dim=0)

    print(f'Memory bank final shape: {memory_bank.shape}')

    return memory_bank

In [ ]:
def build_memory_bank_multiview_cls(dataloader, extractor, device, coreset_ratio):
    """
    Builds a memory bank where each entry = ONE OBJECT (5 views fused).
    """

    memory_bank_list = []

    for imgs, labels, obj_ids in tqdm(dataloader, desc='Extracting MULTIVIEW CLS & coreset'):
        # imgs: [B, 5, 3, 224, 224]
        B, V, C, H, W = imgs.shape

        imgs = imgs.to(device)

        # flatten views → (B*V, 3, 224, 224)
        imgs = imgs.view(B * V, C, H, W)

        feats = extractor(imgs)  # [B*V, D]
        D = feats.shape[-1]

        # reshape back → [B, V, D]
        feats = feats.view(B, V, D)

        # ─────────────────────────────────────────────
        # FUSION: set-based (order-invariant)
        # ─────────────────────────────────────────────
        # option 1: mean pooling (robust baseline)
        feats = feats.mean(dim=1)  # [B, D]

        feats = feats.cpu()

        # coreset on objects (NOT views!)
        K = max(1, int(feats.shape[0] * coreset_ratio))
        idx = torch.randperm(feats.shape[0])[:K]

        memory_bank_list.append(feats[idx])

    memory_bank = torch.cat(memory_bank_list, dim=0)

    print(f'MULTIVIEW CLS memory bank shape: {memory_bank.shape}')

    return memory_bank

In [ ]:
# ── PATCHCORE MEMORY BANKS (HL + LL) ────────────────────────────────────────
memory_bank_hl = build_memory_bank(
    train_dl,
    extractor_hl,
    DEVICE,
    CORESET_RATIO
)

memory_bank_ll = build_memory_bank(
    train_dl,
    extractor_ll,
    DEVICE,
    CORESET_RATIO
)

In [ ]:
# ── SEMANTIC MEMORY BANK (SINGLE-VIEW CLS) ──────────────────────────────────
memory_bank_cls = build_memory_bank_cls(
    semantic_train_dl,
    semantic_extractor,
    DEVICE,
    CORESET_RATIO
)

# ── SEMANTIC MEMORY BANK (MULTIVIEW CLS / OBJECT-LEVEL) ─────────────────────
memory_bank_mv_cls = build_memory_bank_multiview_cls(
    semantic_train_dl,   # NOTA: deve essere il dataset multiview, non dataloader image-level
    semantic_extractor,
    DEVICE,
    CORESET_RATIO
)

# ── PRINT SHAPES ─────────────────────────────────────────────────────────────
print(f'Memory bank HL shape: {memory_bank_hl.shape}')
print(f'Memory bank LL shape: {memory_bank_ll.shape}')
print(f'Memory bank CLS shape: {memory_bank_cls.shape}')
print(f'Memory bank MV-CLS shape: {memory_bank_mv_cls.shape}')


In [ ]:
import os
import torch

# ── OUTPUT DIR ───────────────────────────────────────────────────────────────
OUTPUT_DIR = f'/content/drive/MyDrive/patchcore_output/{CLASS_NAME}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── NEW NAME (2layer + 2semantic) ────────────────────────────────────────────
save_path = os.path.join(OUTPUT_DIR, "patchcore_dinov2_2layer_2semantic.pt")

# ── SAFETY CHECK ─────────────────────────────────────────────────────────────
if os.path.exists(save_path):
    raise FileExistsError(f"Il file {save_path} esiste già! Non verrà sovrascritto.")
else:
    torch.save({
        # ── PATCHCORE (2 LAYERS) ─────────────────────────────────────────────
        "memory_bank_HL": memory_bank_hl,
        "memory_bank_LL": memory_bank_ll,
        "feature_layers_HL": FEATURE_LAYERS_HL,
        "feature_layers_LL": FEATURE_LAYERS_LL,

        # ── SEMANTIC (2 BRANCHES) ────────────────────────────────────────────
        "memory_bank_CLS": memory_bank_cls,
        "memory_bank_MV_CLS": memory_bank_mv_cls,

    }, save_path)

    print(f"File salvato correttamente in {save_path}")

In [ ]:
def compute_anomaly_scores(img_feats, memory_bank, k=1, batch_size=1024):
    P = img_feats.shape[0]
    patch_scores = torch.zeros(P)

    memory_bank = memory_bank.float()

    for start in range(0, P, batch_size):
        end = min(start + batch_size, P)
        q = img_feats[start:end].float()

        dists = torch.cdist(q, memory_bank)  # [bs, M]
        knn_dists = dists.topk(k, dim=1, largest=False).values
        patch_scores[start:end] = knn_dists.mean(dim=1)

    image_score = patch_scores.max().item()
    return patch_scores, image_score


def score_dataloader(dataloader, extractor, memory_bank, device):
    results = []

    for imgs, labels, paths in tqdm(dataloader, desc='Scoring'):
        imgs = imgs.to(device)
        with torch.no_grad():
            feats = extractor(imgs).cpu()  # [B, N_patches, D]
        for i in range(feats.shape[0]):
            ps, im_s = compute_anomaly_scores(feats[i], memory_bank)
            results.append({
                'path': paths[i],
                'label': labels[i].item(),
                'image_score': im_s,
                'patch_scores': ps
            })
    return results


In [ ]:
def patch_scores_to_heatmap(patch_scores, img_size):
    n = patch_scores.shape[0]
    side = int(n ** 0.5)
    patch_scores = patch_scores[:side * side]
    hmap = patch_scores.view(side, side).cpu().numpy()
    hmap = cv2.resize(hmap, (img_size, img_size), interpolation=cv2.INTER_NEAREST)    #cv2.INTER_LINEAR
    #hmap = (hmap - hmap.min()) / (hmap.max() - hmap.min() + 1e-8)
    return hmap



In [ ]:
def compute_anomaly_score_cls(feat, memory_bank, k=1):
    """
    feat: [D]
    memory_bank: [M, D]
    """

    feat = feat.unsqueeze(0).float()
    memory_bank = memory_bank.float()

    dists = torch.cdist(feat, memory_bank)  # [1, M]
    knn_dists = dists.topk(k, dim=1, largest=False).values

    score = knn_dists.mean().item()

    return score
def compute_anomaly_score_mv_cls(feat, memory_bank, k=1):
    """
    feat: [D]  (one object embedding from 5 views)
    memory_bank: [M, D]
    """

    feat = feat.unsqueeze(0).float()
    memory_bank = memory_bank.float()

    dists = torch.cdist(feat, memory_bank)  # [1, M]
    knn_dists = dists.topk(k, dim=1, largest=False).values

    score = knn_dists.mean().item()

    return score

In [ ]:
import os
import torch

SAVE_PATH = f'/content/drive/MyDrive/patchcore_output/{CLASS_NAME}/patchcore_dinov2_2layers.pt'

# Caricamento
if not os.path.exists(SAVE_PATH):
    raise FileNotFoundError(f"Il file {SAVE_PATH} non esiste! Devi prima salvarlo.")

checkpoint = torch.load(SAVE_PATH, map_location=DEVICE)
memory_bank_hl = checkpoint['memory_bank_HL']
memory_bank_ll = checkpoint['memory_bank_LL']
FEATURE_LAYERS_HL = checkpoint['feature_layers_HL']
FEATURE_LAYERS_LL = checkpoint['feature_layers_LL']

print(f"File caricato correttamente da {SAVE_PATH}")

Da qui visualizzazione

In [ ]:
import numpy as np
from collections import deque

def extract_background(image, pad=20, threshold=10):
    # convert PIL → numpy se necessario
    if not isinstance(image, np.ndarray):
        image = np.array(image)
    h, w = image.shape[:2]
    gray = image if len(image.shape) == 2 else image.mean(axis=2)
    # seed mask (cornice)
    seed_mask = np.zeros((h, w), dtype=bool)
    seed_mask[:pad, :] = True
    seed_mask[-pad:, :] = True
    seed_mask[:, :pad] = True
    seed_mask[:, -pad:] = True
    seed_values = gray[seed_mask]
    bg_mean = seed_values.mean()
    visited = np.zeros((h, w), dtype=bool)
    bg_mask = np.zeros((h, w), dtype=bool)
    from collections import deque
    q = deque()
    ys, xs = np.where(seed_mask)
    for i, j in zip(ys, xs):
        q.append((i, j))
        visited[i, j] = True
        bg_mask[i, j] = True
    while q:
        i, j = q.popleft()
        for di, dj in [(-1,0),(1,0),(0,-1),(0,1)]:
            ni, nj = i + di, j + dj
            if 0 <= ni < h and 0 <= nj < w and not visited[ni, nj]:
                if abs(gray[ni, nj] - bg_mean) < threshold:
                    visited[ni, nj] = True
                    bg_mask[ni, nj] = True
                    q.append((ni, nj))

    return bg_mask

import numpy as np
from scipy.ndimage import label as nd_label

def keep_largest_component(bg_mask):
    fg = (~bg_mask).astype(np.uint8)
    labeled, num = nd_label(fg)

    if num == 0:
        return np.zeros_like(bg_mask, dtype=bool)

    sizes = np.bincount(labeled.ravel())
    sizes[0] = 0
    largest_label = np.argmax(sizes)

    out = (labeled == largest_label)

    return ~out

In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import average_precision_score
from math import ceil
from torchvision import transforms

# ── NOME FILE OUTPUT ─────────────────────────────────────────────
OUTPUT_FILE_NAME = "anomaly_visualization_2x2.png"
OUTPUT_PATH = f"/content/drive/MyDrive/patchcore_output/{OUTPUT_FILE_NAME}"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# ── PARAMETRI ─────────────────────────────────────────────────────
IMGS_PER_ROW = 5
IMG_SIZE_PLOT = 224
ALPHA = 0.5

# ── FUNZIONE DENORMALIZZAZIONE ───────────────────────────────────
def denormalize_tensor(img_tensor):
    mean = torch.tensor([0.485, 0.456, 0.406], device=img_tensor.device)[:, None, None]
    std  = torch.tensor([0.229, 0.224, 0.225], device=img_tensor.device)[:, None, None]
    img = img_tensor * std + mean
    img = torch.clamp(img, 0, 1)
    img = (img * 255).byte()
    return transforms.ToPILImage()(img.cpu())

# ── FUNZIONE HEATMAP ─────────────────────────────────────────────
def overlay_heatmap(img_pil, patch_scores, img_size=IMG_SIZE_PLOT, alpha=ALPHA, colormap=cv2.COLORMAP_JET):
    # patch_scores già in [0,1] o qualsiasi scala, non normalizziamo qui
    hmap = patch_scores_to_heatmap(patch_scores, img_size)
    heatmap_color = cv2.applyColorMap((hmap*255).astype(np.uint8), colormap)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(np.array(img_pil.resize((img_size,img_size))), alpha, heatmap_color, 1-alpha, 0)
    return overlay, hmap

# ── PREPARAZIONE FIGURA ──────────────────────────────────────────
n_images = len(test_anom)
n_rows = ceil(n_images / IMGS_PER_ROW) * 3  # 3 righe per GT + HL + LL

fig, axs = plt.subplots(n_rows, IMGS_PER_ROW, figsize=(IMGS_PER_ROW*4, n_rows*4))
axs = np.array(axs)

for idx in range(n_images):
    img_tensor, label, path = test_anom[idx]
    img_pil = denormalize_tensor(img_tensor)
    img_to_plot = np.array(img_pil.resize((IMG_SIZE_PLOT, IMG_SIZE_PLOT)))

    # ── RIGHE E COLONNE ─────────────────────────────
    row_base = (idx // IMGS_PER_ROW) * 3
    col = idx % IMGS_PER_ROW

    # ── GROUND TRUTH SULL'IMMAGINE BASE ─────────────
    rel_dir = os.path.relpath(os.path.dirname(path), os.path.join(BASE_DIR, "train"))
    gt_mask_path = os.path.join(BASE_DIR, "ground_truth_train", rel_dir, os.path.basename(path))

    if os.path.exists(gt_mask_path):
        gt_mask = cv2.imread(gt_mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = (gt_mask > 0).astype(np.uint8)
        gt_mask_resized = cv2.resize(gt_mask, (IMG_SIZE_PLOT, IMG_SIZE_PLOT), interpolation=cv2.INTER_NEAREST)
        overlay_gt = img_to_plot.copy()
        overlay_gt[gt_mask_resized==1] = [255, 0, 0]
        img_base = cv2.addWeighted(img_to_plot, 0.7, overlay_gt, 0.3, 0)
    else:
        img_base = img_to_plot

    axs[row_base, col].imshow(img_base)
    # ── CLS SINGLE-VIEW SCORE ─────────────────────────────
    with torch.no_grad():
        img_tensor_dev = img_tensor.unsqueeze(0).to(DEVICE)
        cls_feat = semantic_extractor(img_tensor_dev)[0]  # [D]
        cls_feat = cls_feat.cpu()
        score_cls = compute_anomaly_score_cls(cls_feat, memory_bank_cls)

    # ── GT + CLS SCORE LABEL ──────────────────────────────
    axs[row_base, col].set_title(f"CLS: {score_cls:.3f}")
    axs[row_base, col].axis('off')

    # ── ESTRAI FEATURE HL/LL ─────────────────────────
    img_tensor = img_tensor.unsqueeze(0).to(DEVICE)
    feats_hl = extractor_hl(img_tensor)[0]
    feats_ll = extractor_ll(img_tensor)[0]

    if memory_bank_hl.device.type == "cpu":
        feats_hl = feats_hl.cpu()
        feats_ll = feats_ll.cpu()

    # ── PATCH SCORES HL/LL ──────────────────────────
    patch_scores_hl, _ = compute_anomaly_scores(feats_hl, memory_bank_hl)
    patch_scores_ll, _ = compute_anomaly_scores(feats_ll, memory_bank_ll)


    # ── HEATMAPS HL/LL (SENZA GT) ──────────────────
    overlay_hl, hmap_hl = overlay_heatmap(img_pil, patch_scores_hl)
    overlay_ll, hmap_ll = overlay_heatmap(img_pil, patch_scores_ll)

    # ── CALCOLO AP PIXEL-WISE ──────────────────────
    if os.path.exists(gt_mask_path):
        gt_resized = cv2.resize(gt_mask, (hmap_hl.shape[1], hmap_hl.shape[0]), interpolation=cv2.INTER_NEAREST)
        ap_hl = average_precision_score(gt_resized.flatten(), hmap_hl.flatten()) if gt_resized.sum()>0 else 0.0
        ap_ll = average_precision_score(gt_resized.flatten(), hmap_ll.flatten()) if gt_resized.sum()>0 else 0.0
    else:
        ap_hl = ap_ll = 0.0

    axs[row_base+1, col].imshow(overlay_hl)
    axs[row_base+1, col].set_title(f"HL heatmap\nAP: {ap_hl:.3f}")
    axs[row_base+1, col].axis('off')

    axs[row_base+2, col].imshow(overlay_ll)
    axs[row_base+2, col].set_title(f"LL heatmap\nAP: {ap_ll:.3f}")
    axs[row_base+2, col].axis('off')

# ── DISABILITA ASSI EXTRA ─────────────────────────
for r in range(n_rows):
    for c in range(IMGS_PER_ROW):
        if r*IMGS_PER_ROW + c >= n_images:
            axs[r, c].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_PATH, dpi=150)
print(f"Visualizzazione salvata in {OUTPUT_PATH}")
plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image


# ─────────────────────────────────────────────────────────────
# MV-CLS SCORE
# ─────────────────────────────────────────────────────────────
def compute_mv_cls_score(imgs, semantic_extractor, memory_bank_mv_cls, device):

    semantic_extractor.eval()

    batch = []

    for img in imgs:
        if isinstance(img, Image.Image):
            img = transform(img)
        elif isinstance(img, torch.Tensor):
            pass
        else:
            raise TypeError("Input must be PIL image or tensor")

        batch.append(img)

    batch = torch.stack(batch).to(device)  # [5, 3, H, W]

    with torch.no_grad():
        feats = semantic_extractor(batch)     # [5, D]
        feats = feats.mean(dim=0)             # MV fusion → [D]

    feats = feats.unsqueeze(0).to(device).float()
    mb = memory_bank_mv_cls.to(device).float()

    dists = torch.cdist(feats, mb)
    score = dists.min(dim=1).values.item()

    return score


# ─────────────────────────────────────────────────────────────
# VISUALIZZAZIONE
# ─────────────────────────────────────────────────────────────
def show_multiview_with_mvcls(imgs, score):

    fig, axs = plt.subplots(1, 5, figsize=(15, 3))

    titles = ["front", "back-down", "back-left", "back-up", "back-right"]

    for i in range(5):
        axs[i].imshow(imgs[i])
        axs[i].set_title(titles[i])
        axs[i].axis("off")

    fig.suptitle(f"MV-CLS SCORE: {score:.4f}", fontsize=14)

    plt.tight_layout()
    plt.show()


# ─────────────────────────────────────────────────────────────
# PIPELINE COMPLETA
# ─────────────────────────────────────────────────────────────
def run_mvcls_test(img_paths, semantic_extractor, memory_bank_mv_cls, device):

    imgs = [Image.open(p).convert("RGB") for p in img_paths]

    score = compute_mv_cls_score(
        imgs,
        semantic_extractor,
        memory_bank_mv_cls,
        device
    )

    show_multiview_with_mvcls(imgs, score)

    return score

In [ ]:
print(TEST_ANOM_DIRS)

In [ ]:
01 - img_00440512f31f25c7d053_view1.png
02 - img_00440512f31f25c7d053_view2.png
03 - img_00440512f31f25c7d053_view3.png
04 - img_00440512f31f25c7d053_view4.png
05 - img_00440512f31f25c7d053_view5.png
06 - img_019ff3b1495481a1ffb1_view1.png
07 - img_019ff3b1495481a1ffb1_view2.png
08 - img_019ff3b1495481a1ffb1_view3.png
09 - img_019ff3b1495481a1ffb1_view4.png
10 - img_019ff3b1495481a1ffb1_view5.png
11 - img_01a2e20d6f95662a5fad_view1.png
12 - img_01a2e20d6f95662a5fad_view2.png
13 - img_01a2e20d6f95662a5fad_view3.png
14 - img_01a2e20d6f95662a5fad_view4.png
15 - img_01a2e20d6f95662a5fad_view5.png
16 - img_0322576ab6f0c9b18e91_view1.png
17 - img_0322576ab6f0c9b18e91_view2.png
18 - img_0322576ab6f0c9b18e91_view3.png
19 - img_0322576ab6f0c9b18e91_view4.png
20 - img_0322576ab6f0c9b18e91_view5.png

In [ ]:
#os.path.join(TRAIN_GOOD_DIR, "img_0b2aeb021e24a7f03e92_view5.png"),
#os.path.join('/content/drive/MyDrive/dataset_ad/class_08/class_08/train/anomaly_03', "img_723b766aacd3f6e3d500_view1.png"),
#os.path.join('/content/drive/MyDrive/dataset_ad/class_02/class_02/train/anomaly_01', "img_ac774fb0858481e6f65e_view4.png"),
img_paths = [
    os.path.join('/content/drive/MyDrive/dataset_ad/class_02/class_02/train/good', "img_00440512f31f25c7d053_view1.png"),
    os.path.join('/content/drive/MyDrive/dataset_ad/class_02/class_02/train/good', "img_01a2e20d6f95662a5fad_view2.png"),
    os.path.join('/content/drive/MyDrive/dataset_ad/class_02/class_02/train/good', "img_01a2e20d6f95662a5fad_view3.png"),
    os.path.join('/content/drive/MyDrive/dataset_ad/class_02/class_02/train/good', "img_01a2e20d6f95662a5fad_view4.png"),
    os.path.join('/content/drive/MyDrive/dataset_ad/class_02/class_02/train/good', "img_01a2e20d6f95662a5fad_view5.png")
]

score = run_mvcls_test(
    img_paths,
    semantic_extractor,
    memory_bank_mv_cls,
    DEVICE
)

print("MV-CLS SCORE:", score)